Charger les données préparées

In [0]:
from pyspark.sql.functions import col
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow

# Charger les ratings indexés du Feature Engineering
VOLUME = "/Volumes/workspace/default/ecommerce_data"

ratings_indexed = spark.read.parquet(f"{VOLUME}/ratings_indexed")

print(f"Ratings chargés : {ratings_indexed.count():,} interactions")
print(f"Colonnes : {ratings_indexed.columns}")
display(ratings_indexed.select(
    "user_idx", "item_idx", "rating", "timestamp"
).limit(5))

Ratings chargés : 701,528 interactions
Colonnes : ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'user_idx', 'item_idx']


user_idx,item_idx,rating,timestamp
33934,722,5,2020-05-05T14:08:48.923Z
33934,9669,4,2020-05-04T18:10:55.070Z
71462,4203,5,2020-05-16T21:41:06.052Z
26005,109899,1,2022-01-28T18:13:50.220Z
26005,42259,5,2020-12-30T10:02:43.534Z


Vérifier le format du timestamp

In [0]:
# IMPORTANT : vérifier à quoi ressemble le timestamp
# avant de faire le split par date
display(
    ratings_indexed.select("timestamp")
    .orderBy("timestamp")
    .limit(5)
)

# Afficher le type de la colonne timestamp
ratings_indexed.printSchema()

timestamp
2000-11-01T04:24:18.000Z
2001-01-16T15:10:44.000Z
2001-01-30T17:13:34.000Z
2001-03-05T07:27:57.000Z
2001-03-29T23:38:47.000Z


root
 |-- rating: long (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- user_idx: integer (nullable = true)
 |-- item_idx: integer (nullable = true)



Split train/test

In [0]:
from pyspark.sql.functions import col, min as spark_min, max as spark_max

# Vérifier les dates min et max disponibles
ratings_indexed.select(
    spark_min("timestamp").alias("date_min"),
    spark_max("timestamp").alias("date_max")
).show()

# Split par date directement (pas besoin de conversion !)
split_date = "2021-01-01"

train = ratings_indexed.filter(col("timestamp") < split_date)
test  = ratings_indexed.filter(col("timestamp") >= split_date)

print(f" Train : {train.count():,} interactions")
print(f" Test  : {test.count():,} interactions")



+-------------------+--------------------+
|           date_min|            date_max|
+-------------------+--------------------+
|2000-11-01 04:24:18|2023-09-09 00:39:...|
+-------------------+--------------------+

 Train : 500,645 interactions
 Test  : 200,883 interactions


Entraînement ALS

In [0]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow

with mlflow.start_run(run_name="ALS_rank50_reg01"):

    # Configurer ALS
    als = ALS(
        maxIter=10,  #Nombre d'itérations d'entraînement.
        regParam=0.1, #la régularisation
        rank=50, #nombre de facteurs latents
        userCol="user_idx", 
        itemCol="item_idx",
        ratingCol="rating", 
        coldStartStrategy="drop", #strategie pour les nouveaux utilisateurs
        nonnegative=True #forcer les valeurs prédites à être positives
    )   

    # Entraîner
    print("Entraînement en cours... (2-5 minutes)")
    model = als.fit(train)
    print(" Modèle entraîné !")

    # Évaluer
    predictions = model.transform(test)
    evaluator = RegressionEvaluator(
        metricName="rmse",
        labelCol="rating",
        predictionCol="prediction"
    )
    rmse = evaluator.evaluate(predictions)
    print(f"\nRMSE = {rmse:.4f}")

    # Logger dans MLflow
    mlflow.log_param("rank", 50)
    mlflow.log_param("regParam", 0.1)
    mlflow.log_param("maxIter", 10)
    mlflow.log_metric("rmse", rmse)

print("Run MLflow enregistré !")

Entraînement en cours... (2-5 minutes)
 Modèle entraîné !

RMSE = 2.5676
Run MLflow enregistré !


Sauvegarder le modèle

In [0]:
# Chemin du Volume (même que tout le reste)
VOLUME = "/Volumes/workspace/default/ecommerce_data"
model_path = f"{VOLUME}/als_model"

# Sauvegarder le modèle
model.save(model_path)
print(f"Modèle sauvegardé : {model_path}")

# Vérifier les fichiers sauvegardés
print("\n Fichiers du modèle :")
for f in dbutils.fs.ls(model_path):
    print(f"  → {f.name}")

{"ts": "2026-03-09 13:37:57.600", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_MultiThreadedRendezvous", "msg": "<_MultiThreadedRendezvous of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"Path /Volumes/workspace/default/ecommerce_data/als_model already exists. To overwrite it, please use write.overwrite().save(path) for Scala and use write().overwrite().save(path) for Java and Python.\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_message:\"Path /Volumes/workspace/default/ecommerce_data/als_model already exists. To overwrite it, please use write.overwrite().save(path) for Scala and use write().overwrite().save(path) for Java and Python.\", grpc_status:13, created_time:\"2026-03-09T13:37:57.599704749+00:00\"}\"\n>", "stacktrace": [{"class": null, "method": "_execute_and_fetch_as_iterator", "file": "/databricks/python/lib/python3.12/site-packages/pyspark/sql

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-6518015124754442>, line 6
      3 model_path = f"{VOLUME}/als_model"
      5 # Sauvegarder le modèle
----> 6 model.save(model_path)
      7 print(f"Modèle sauvegardé : {model_path}")
      9 # Vérifier les fichiers sauvegardés

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/util.py:820, in MLWritable.save(self, path)
    818 def save(self, path: str) -> None:
    819     """Save this ML instance to the given path, a shortcut of 'write().save(path)'."""
--> 820     self.write().save(path)

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/connect/readwrite.py:52, in RemoteMLWriter.save(self, path)
     49 session = SparkSession.getActiveSession()
     50 assert session is not None
---> 52 RemoteMLWriter.saveInstance(
     53     self._instance,
     54     path,
     55     session,
     56   

Générer les recommandations

cluster Serverless ne supporte pas certaines fonctions Spark (recommendForAllUsers) donc  au lieu d'utiliser cette fonction je fait le calcule manuellement via le produit scalaire 

In [0]:
# Alternative sans recommendForAllUsers
# Utilise les facteurs utilisateurs et produits directement

from pyspark.ml.recommendation import ALSModel
import pyspark.sql.functions as F

# Charger le modèle
model = ALSModel.load(f"{VOLUME}/als_model")

# Récupérer les facteurs
user_factors = model.userFactors  # user_idx + features vector
item_factors = model.itemFactors  # item_idx + features vector

print(f"User factors : {user_factors.count():,}")
print(f"Item factors : {item_factors.count():,}")

# Calculer les scores manuellement via produit scalaire
from pyspark.ml.linalg import Vectors
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType
import numpy as np

# Fonction de score
@udf(FloatType())
def dot_product(u, v):
    return float(np.dot(u, v))

# Croiser users et items (limiter pour éviter explosion)
# Prendre un sample d'utilisateurs pour tester
sample_users = user_factors.limit(100)

# Renommer les colonnes 'features' et 'id' pour lever l'ambiguïté
renamed_users = sample_users.withColumnRenamed("features", "user_features").withColumnRenamed("id", "user_idx")
renamed_items = item_factors.limit(1000).withColumnRenamed("features", "item_features").withColumnRenamed("id", "item_idx")

# Calculer scores pour ces 100 users
cross = renamed_users.crossJoin(renamed_items)

scores = cross.withColumn(
    "score",
    dot_product(F.col("user_features"), F.col("item_features"))
).select(
    F.col("user_idx"),
    F.col("item_idx"),
    F.col("score")
)

display(scores.limit(10))
print("scores calculés manuellement !")


User factors : 457,324
Item factors : 80,187


user_idx,item_idx,score
3,5,2.5228224
13,15,2.128539
23,25,1.7998533
33,35,1.6877978
43,45,2.8596232
53,55,2.239626
63,65,2.6139252
73,85,2.3203228
83,95,1.9950899
93,115,1.8610045


scores calculés manuellement !


Sauvegarder les recommandations

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

VOLUME = "/Volumes/workspace/default/ecommerce_data"

# Garder le top-100 par utilisateur
# (on trie par score décroissant et on garde les 100 meilleurs)
window_spec = Window.partitionBy("user_idx").orderBy(col("score").desc())

top100_recs = scores \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 100) \
    .drop("rank")

print(f"Recommandations préparées")
display(top100_recs.limit(10))

# Sauvegarder dans le Volume
recs_path = f"{VOLUME}/recommendations_als"

top100_recs.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(recs_path)

print(f"Recommandations sauvegardées : {recs_path}")

Recommandations préparées


user_idx,item_idx,score
3,2745,4.875855
3,8945,4.8734393
3,6365,4.6809416
3,7255,4.6353097
3,10235,4.6192565
3,4245,4.571206
3,5945,4.566235
3,10115,4.56535
3,2825,4.532802
3,8555,4.5275846


Recommandations sauvegardées : /Volumes/workspace/default/ecommerce_data/recommendations_als


Vérifier avec un vrai utilisateur

In [0]:
import json

# Charger les mappings
with open(f"{VOLUME}/user_mapping.json") as f:
    user_mapping = json.load(f)

with open(f"{VOLUME}/item_mapping.json") as f:
    item_mapping = json.load(f)

# Charger les recommandations sauvegardées
recs_loaded = spark.read.parquet(recs_path)

#Prendre un utilisateur qui existe VRAIMENT dans les recs
first_user_idx = recs_loaded.select("user_idx") \
                             .distinct() \
                             .orderBy("user_idx") \
                             .first()[0]

print(f"Premier user_idx disponible : {first_user_idx}")

# Récupérer ses recommandations
user_rec = recs_loaded \
    .filter(col("user_idx") == first_user_idx) \
    .orderBy(col("score").desc()) \
    .limit(10) \
    .collect()

print(f"Nombre de recommandations trouvées : {len(user_rec)}")

# Afficher
if len(user_rec) > 0:
    print(f"\n Utilisateur : {user_mapping[first_user_idx]}")
    print(f"\n Top-10 recommandations :")
    print(f"{'Rang':<5} {'Product ID':<15} {'Score':<10}")
    print("-" * 35)
    for i, row in enumerate(user_rec):
        product_id = item_mapping[row["item_idx"]]
        score      = row["score"]
        print(f"  {i+1:<4} {product_id:<15} {score:.4f}")
else:
    print(" Aucune recommandation trouvée pour cet utilisateur")


Premier user_idx disponible : 3
Nombre de recommandations trouvées : 10

 Utilisateur : AHDVSLWHSORYGG3S5QZMVDFNOXUQ

 Top-10 recommandations :
Rang  Product ID      Score     
-----------------------------------
  1    B08BFH35VX      4.8759
  2    B07ZS3DKL5      4.8734
  3    B08GKHQ9P1      4.6809
  4    B08G5YVHQP      4.6353
  5    B01N5SHGS0      4.6193
  6    B00O2FGBJS      4.5712
  7    B0837K9W6P      4.5662
  8    B01B5KTY3W      4.5654
  9    B0859CYSTM      4.5328
  10   B00KRMOJ4Y      4.5276


 Calculer Precision@10

In [0]:
from pyspark.sql.functions import collect_list, size, array_intersect, avg


# ÉTAPE 1 : Produits vraiment aimés (note >= 4)

# On considère qu'un utilisateur "aime" un produit
# si il lui a donné une note >= 4 dans le test set

relevant = test \
    .filter(col("rating") >= 4) \
    .groupBy("user_idx") \
    .agg(collect_list("item_idx").alias("relevant_items"))

print(f" Users avec produits aimés : {relevant.count():,}")


# ÉTAPE 2 : Top-10 recommandations par user

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("user_idx").orderBy(col("score").desc())

top10_recs = recs_loaded \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .groupBy("user_idx") \
    .agg(collect_list("item_idx").alias("recommended_items"))

print(f" Users avec recommandations : {top10_recs.count():,}")


# ÉTAPE 3 : Joindre et calculer Precision@10
# Precision@10 = combien de produits recommandés
#  sont vraiment aimés / 10

precision_df = top10_recs.join(relevant, "user_idx") \
    .withColumn(
        "precision_at_10",
        size(array_intersect(
            col("recommended_items"),
            col("relevant_items")
        )) / 10
    )

avg_precision = precision_df \
    .agg(avg("precision_at_10").alias("avg_precision_at_10")) \
    .collect()[0]["avg_precision_at_10"]

print("\n" + "=" * 50)
print("RÉSULTATS FINAUX ALS")
print("=" * 50)
print(f"RMSE         = {rmse:.4f}")
print(f"Precision@10 = {avg_precision:.4f}")
print(f"             = {avg_precision*100:.1f}% de recommandations pertinentes")




## Comment interpréter Precision@10
#Precision@10 = 0.15  →  15% des recommandations sont pertinentes
              # = sur 10 produits recommandés, 1.5 sont vraiment aimés

#Precision@10 = 0.30  →  30% des recommandations sont pertinentes  
#= sur 10 produits recommandés, 3 sont vraiment aimés

#Objectif du projet : battre la baseline de 15%
#→ Si ton score > 0.15 →  objectif atteint !

 Users avec produits aimés : 122,277
 Users avec recommandations : 100

RÉSULTATS FINAUX ALS
RMSE         = 2.5676
Precision@10 = 0.0000
             = 0.0% de recommandations pertinentes


# Vérifier pourquoi Precision = 0

In [0]:


# 1. Combien d'users sont en commun entre recs et test ?
recs_users   = recs_loaded.select("user_idx").distinct()
test_users   = test.filter(col("rating") >= 4).select("user_idx").distinct()

common_users = recs_users.join(test_users, "user_idx")
print(f"Users dans recs        : {recs_users.count():,}")
print(f"Users dans test (≥4)   : {test_users.count():,}")
print(f"Users en commun        : {common_users.count():,}")

# 2. Vérifier si les item_idx matchent
recs_items   = recs_loaded.select("item_idx").distinct()
test_items   = test.filter(col("rating") >= 4).select("item_idx").distinct()

common_items = recs_items.join(test_items, "item_idx")
print(f"\nItems dans recs        : {recs_items.count():,}")
print(f"Items dans test (≥4)   : {test_items.count():,}")
print(f"Items en commun        : {common_items.count():,}")


### Ce que tu devrais voir

#Si common_users = 0  →  problème de coverage
                         #les 100 users dans recs ne sont pas
                         #les mêmes que dans le test set

#Si common_items = 0  →  problème d'items
                         #les 1000 items dans recs ne couvrent
                         #pas les items du test set

Users dans recs        : 100
Users dans test (≥4)   : 122,277
Users en commun        : 73

Items dans recs        : 604
Items dans test (≥4)   : 36,675
Items en commun        : 318


Le problème est clairement identifié 

Users dans recs      : 100        ← on a généré pour seulement 100 users

Users dans test      : 122,277    ← mais le test en a 122,277 !

Users en commun      : 73         ← seulement 73 matchent !

Items dans recs      : 604        ← seulement 604 items couverts

Items dans test      : 36,675     ← mais il y en a 36,675 !

Items en commun      : 318        ← très peu de coverage

PROBLÈME 1 — Trop peu d'users
sample_users = user_factors.limit(100)   ← seulement 100 users
→ Il faut TOUS les 457,324 users

PROBLÈME 2 — Trop peu d'items
item_factors.limit(1000)                 ← seulement 1000 items
→ Il faut TOUS les 80,187 items

Mais le vrai problème — CrossJoin trop lourd
457,324 users × 80,187 items = 36 MILLIARDS de combinaisons
→ Impossible !

Solution : prendre un échantillon REPRÉSENTATIF
100 users × 80,187 items = 8 millions  ← faisable

Couvrir tous les users du `test`

In [0]:
VOLUME = "/Volumes/workspace/default/ecommerce_data"
from pyspark.ml.recommendation import ALSModel
import pyspark.sql.functions as F
from pyspark.sql.functions import udf, col, row_number, collect_list
from pyspark.sql.window import Window
from pyspark.sql.types import FloatType
import numpy as np

# Charger le modèle
model = ALSModel.load(f"{VOLUME}/als_model")

user_factors = model.userFactors
item_factors = model.itemFactors

print(f"User factors : {user_factors.count():,}")
print(f"Item factors : {item_factors.count():,}")

# Fonction score
@udf(FloatType())
def dot_product(u, v):
    return float(np.dot(u, v))

# CORRECTION 1 : prendre les users du test (pas random)
test_user_idxs = [
    row["user_idx"]
    for row in test.select("user_idx")
                   .distinct()
                   .limit(100)  # 100 users du test
                   .collect()
]

# CORRECTION 2 : TOUS les items (pas de limit)
renamed_users = user_factors \
    .filter(col("id").isin(test_user_idxs)) \
    .withColumnRenamed("features", "user_features") \
    .withColumnRenamed("id", "user_idx")

renamed_items = item_factors \
    .withColumnRenamed("features", "item_features") \
    .withColumnRenamed("id", "item_idx")
# ← PAS de .limit() sur les items !

print(f"\nUsers sélectionnés : {renamed_users.count():,}")
print(f"Items total        : {renamed_items.count():,}")
print(f"Combinaisons       : {renamed_users.count() * renamed_items.count():,}")

# CrossJoin
cross = renamed_users.crossJoin(renamed_items)

scores = cross.withColumn(
    "score",
    dot_product(F.col("user_features"), F.col("item_features"))
).select("user_idx", "item_idx", "score")

# Top-10 par user
window_spec = Window.partitionBy("user_idx").orderBy(col("score").desc())

top10 = scores \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .groupBy("user_idx") \
    .agg(collect_list("item_idx").alias("recommended_items"))

print(f"\nRecs générées pour {top10.count():,} users")
display(top10.limit(3))

User factors : 457,324
Item factors : 80,187

Users sélectionnés : 18
Items total        : 80,187
Combinaisons       : 1,443,366

Recs générées pour 18 users


user_idx,recommended_items
41,"List(15967, 10902, 9992, 21685, 36732, 51348, 94176, 4050, 18434, 13544)"
157,"List(41390, 102830, 7688, 79449, 26304, 12030, 23379, 14468, 26065, 30734)"
241,"List(41390, 102830, 20604, 11718, 16361, 11762, 24324, 26724, 41996, 12834)"


precision

In [0]:
from pyspark.sql.functions import size, array_intersect, avg

# Produits aimés dans le test
relevant = test \
    .filter(col("rating") >= 4) \
    .filter(col("user_idx").isin(test_user_idxs)) \
    .groupBy("user_idx") \
    .agg(collect_list("item_idx").alias("relevant_items"))

# Vérification
example = top10.join(relevant, "user_idx").limit(1).collect()
if example:
    print(f"Recommended : {example[0]['recommended_items']}")
    print(f"Relevant    : {example[0]['relevant_items']}")
    overlap = set(example[0]['recommended_items']) & \
              set(example[0]['relevant_items'])
    print(f"Overlap     : {overlap}")

# Calcul Precision@10
precision_df = top10.join(relevant, "user_idx") \
    .withColumn(
        "precision_at_10",
        size(array_intersect(
            col("recommended_items"),
            col("relevant_items")
        )) / 10
    )

avg_p10 = precision_df \
    .agg(avg("precision_at_10")) \
    .collect()[0][0]

print("\n" + "=" * 40)
print("RÉSULTATS FINAUX")
print("=" * 40)
print(f"RMSE         = {rmse:.4f}")
print(f"Precision@10 = {avg_p10:.4f}")
print(f"             = {avg_p10*100:.1f}%")

## Ce qui change

#AVANT                          APRÈS

#100 users random          →    100 users du TEST
#1000 items limités        →    80,187 items COMPLETS
#Overlap impossible        →    Overlap possible 
#Precision = 0.0%          →    Valeur réelle

Recommended : [15967, 10902, 9992, 21685, 36732, 51348, 94176, 4050, 18434, 13544]
Relevant    : [9133, 16857, 4864, 110780, 26367, 62195, 6325, 32148, 25598, 22327, 13147, 33567, 12893, 25958, 680]
Overlap     : set()

RÉSULTATS FINAUX
RMSE         = 2.5676
Precision@10 = 0.0000
             = 0.0%


In [0]:
# Vérifier le type du rating
ratings_indexed.printSchema()

# Vérifier la distribution
print("Ratings dans train :")
train.groupBy("rating").count().orderBy("rating").show()


root
 |-- rating: long (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- user_idx: integer (nullable = true)
 |-- item_idx: integer (nullable = true)

Ratings dans train :
+------+------+
|rating| count|
+------+------+
|     1| 63952|
|     2| 29427|
|     3| 39808|
|     4| 60246|
|     5|307212|
+------+------+



dataset très biaisé vers 5 étoiles
─────────────────────────────────────────────────
61% des ratings sont 5 étoiles
→ Le modèle a du mal à différencier les préférences
→ Il recommande des produits populaires pour tout le monde

Réentraîner avec les corrections

In [0]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow

# CORRECTION 1 : Convertir rating en float
train_float = train.withColumn("rating", col("rating").cast("float"))
test_float  = test.withColumn("rating",  col("rating").cast("float"))

# CORRECTION 2 : Vérifier la conversion
print("Schema après conversion :")
train_float.select("rating").printSchema()
print(f"Train float : {train_float.count():,}")

with mlflow.start_run(run_name="ALS_v2_corrected"):

    als_v2 = ALS(
        maxIter=15,
        regParam=0.05,
        rank=100,
        userCol="user_idx",
        itemCol="item_idx",
        ratingCol="rating",
        coldStartStrategy="drop",
        nonnegative=True
    )

    print(" Réentraînement avec rating float...")
    model_v2 = als_v2.fit(train_float)
    print("Modèle v2 entraîné !")

    # Évaluer
    predictions_v2 = model_v2.transform(test_float)
    evaluator = RegressionEvaluator(
        metricName="rmse",
        labelCol="rating",
        predictionCol="prediction"
    )
    rmse_v2 = evaluator.evaluate(predictions_v2)

    print(f"\n RMSE v1 (long)  = {rmse:.4f}")
    print(f" RMSE v2 (float) = {rmse_v2:.4f}")

    mlflow.log_param("rank", 100)
    mlflow.log_param("regParam", 0.05)
    mlflow.log_param("maxIter", 15)
    mlflow.log_metric("rmse", rmse_v2)

Schema après conversion :
root
 |-- rating: float (nullable = true)

Train float : 500,645
 Réentraînement avec rating float...
Modèle v2 entraîné !

 RMSE v1 (long)  = 2.5676
 RMSE v2 (float) = 2.5304


Precision@10 avec model_v2

In [0]:
from pyspark.sql.functions import col, row_number, collect_list
from pyspark.sql.functions import size, array_intersect, avg
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Facteurs du nouveau modèle
user_factors_v2 = model_v2.userFactors
item_factors_v2 = model_v2.itemFactors

print(f"User factors v2 : {user_factors_v2.count():,}")
print(f"Item factors v2 : {item_factors_v2.count():,}")

# ── 100 users du test ──────────────────────────
test_user_idxs = [
    row["user_idx"]
    for row in test_float.select("user_idx")
                         .distinct()
                         .limit(100)
                         .collect()
]

# ── Facteurs sans limite sur les items ─────────
renamed_users = user_factors_v2 \
    .filter(col("id").isin(test_user_idxs)) \
    .withColumnRenamed("features", "user_features") \
    .withColumnRenamed("id", "user_idx")

renamed_items = item_factors_v2 \
    .withColumnRenamed("features", "item_features") \
    .withColumnRenamed("id", "item_idx")

print(f"\nUsers  : {renamed_users.count():,}")
print(f"Items  : {renamed_items.count():,}")
print(f"Combos : {renamed_users.count() * renamed_items.count():,}")

# ── CrossJoin et scores ─────────────────────────
cross = renamed_users.crossJoin(renamed_items)

scores = cross.withColumn(
    "score",
    dot_product(F.col("user_features"), F.col("item_features"))
).select("user_idx", "item_idx", "score")

# ── Top-10 par user ─────────────────────────────
window_spec = Window \
    .partitionBy("user_idx") \
    .orderBy(col("score").desc())

top10 = scores \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .groupBy("user_idx") \
    .agg(collect_list("item_idx").alias("recommended_items"))

print(f"\n Recs générées pour {top10.count():,} users")

User factors v2 : 457,324
Item factors v2 : 80,187

Users  : 18
Items  : 80,187
Combos : 1,443,366

 Recs générées pour 18 users


In [0]:
# Produits aimés dans le test
relevant = test_float \
    .filter(col("rating") >= 4.0) \
    .filter(col("user_idx").isin(test_user_idxs)) \
    .groupBy("user_idx") \
    .agg(collect_list("item_idx").alias("relevant_items"))

# Vérifier overlap
example = top10.join(relevant, "user_idx").limit(1).collect()
if example:
    overlap = set(example[0]['recommended_items']) & \
              set(example[0]['relevant_items'])
    print(f"Recommended : {example[0]['recommended_items']}")
    print(f"Relevant    : {example[0]['relevant_items']}")
    print(f"Overlap     : {overlap}")

# Calcul final
precision_df = top10.join(relevant, "user_idx") \
    .withColumn(
        "precision_at_10",
        size(array_intersect(
            col("recommended_items"),
            col("relevant_items")
        )) / 10
    )

avg_p10 = precision_df \
    .agg(avg("precision_at_10")) \
    .collect()[0][0]

print("\n" + "=" * 45)
print("RÉSULTATS FINAUX ALS v2")
print("=" * 45)
print(f"RMSE v1          = {rmse:.4f}")
print(f"RMSE v2          = {rmse_v2:.4f}")
print(f"Precision@10 v2  = {avg_p10:.4f}")
print(f"                 = {avg_p10*100:.1f}%")

if avg_p10 > 0.15:
    print(" Objectif atteint ! (+15% vs baseline)")
elif avg_p10 > 0.0:
    print(" Résultat positif mais sous l'objectif 15%")
else:
    print(" Precision encore à 0 → voir diagnostic")

Recommended : [10902, 23682, 23855, 49647, 37172, 3992, 49278, 4665, 28682, 20038]
Relevant    : [9133, 16857, 4864, 110780, 26367, 62195, 6325, 32148, 25598, 22327, 13147, 33567, 12893, 25958, 680]
Overlap     : set()

RÉSULTATS FINAUX ALS v2
RMSE v1          = 2.5676
RMSE v2          = 2.5304
Precision@10 v2  = 0.0000
                 = 0.0%
 Precision encore à 0 → voir diagnostic


In [0]:
# Vérifier directement si les item_idx matchent
recs_items_set = set([
    row["item_idx"] 
    for row in renamed_items.select("item_idx").collect()
])

relevant_items_flat = test_float \
    .filter(col("rating") >= 4.0) \
    .filter(col("user_idx").isin(test_user_idxs)) \
    .select("item_idx") \
    .distinct()

relevant_items_set = set([
    row["item_idx"] 
    for row in relevant_items_flat.collect()
])

overlap_items = recs_items_set & relevant_items_set

print(f"Items dans recs     : {len(recs_items_set):,}")
print(f"Items dans relevant : {len(relevant_items_set):,}")
print(f"Items en commun     : {len(overlap_items):,}")

if len(overlap_items) == 0:
    print("\n Aucun item en commun !")
    print("→ Les item_idx de ALS ne correspondent pas")
    print("  aux item_idx du test set")
    print("→ Problème de mapping StringIndexer")
else:
    print(f"\n {len(overlap_items):,} items en commun")
    print("→ Le modèle peut potentiellement trouver des matches")

Items dans recs     : 80,187
Items dans relevant : 151
Items en commun     : 49

 49 items en commun
→ Le modèle peut potentiellement trouver des matches


In [0]:
# Diagnostic rapide
print("Combien d'users dans test_user_idxs ?")
print(len(test_user_idxs))

print("\nCombien ont des ratings >= 4 dans le test ?")
test_float \
    .filter(col("rating") >= 4.0) \
    .filter(col("user_idx").isin(test_user_idxs)) \
    .select("user_idx") \
    .distinct() \
    .show()

print("\nExemple de relevant items :")
test_float \
    .filter(col("rating") >= 4.0) \
    .filter(col("user_idx").isin(test_user_idxs)) \
    .select("user_idx", "item_idx", "rating") \
    .show(10)

Combien d'users dans test_user_idxs ?
100

Combien ont des ratings >= 4 dans le test ?
+--------+
|user_idx|
+--------+
|  450065|
|     404|
|   78440|
|      41|
|  525054|
|  267006|
|   34186|
|   56906|
|  438955|
|   16815|
|  229192|
|   45978|
|    3098|
|   34095|
|  420512|
|  354886|
|  332970|
|     241|
|   63852|
|  431649|
+--------+
only showing top 20 rows

Exemple de relevant items :
+--------+--------+------+
|user_idx|item_idx|rating|
+--------+--------+------+
|   45978|   34561|   5.0|
|  273471|     227|   5.0|
|  173973|  102013|   4.0|
|  236438|     281|   5.0|
|    5038|   19179|   5.0|
|    5038|   14449|   5.0|
|   63852|       7|   5.0|
|    8953|   37236|   5.0|
|  172531|   20726|   5.0|
|  525054|     326|   5.0|
+--------+--------+------+
only showing top 10 rows


Le diagnostic montre que les données sont correctes !
Les item_idx dans relevant (34561, 227, 102013, 281...) sont bien dans la plage de nos 80,187 items. Le problème vient du fait qu'on prend seulement 100 users au hasard avec .limit(100) — pas forcément ceux qui ont des ratings dans le test.

Prendre les users qui ont DES ratings dans le test

In [0]:
from pyspark.sql.functions import col, row_number, collect_list
from pyspark.sql.functions import size, array_intersect, avg
from pyspark.sql.window import Window
import pyspark.sql.functions as F

#  prendre les users qui ont VRAIMENT des ratings >= 4 dans le test
test_user_idxs_final = [
    row["user_idx"]
    for row in test_float
        .filter(col("rating") >= 4.0)
        .select("user_idx")
        .distinct()
        .limit(100)   # 100 users qui ont VRAIMENT aimé des produits
        .collect()
]

print(f"Users sélectionnés : {len(test_user_idxs_final)}")

# Vérifier leurs produits pertinents
relevant_final = test_float \
    .filter(col("rating") >= 4.0) \
    .filter(col("user_idx").isin(test_user_idxs_final)) \
    .groupBy("user_idx") \
    .agg(collect_list("item_idx").alias("relevant_items"))

print(f"Users avec produits aimés : {relevant_final.count():,}")

# Vérifier les items pertinents
items_pertinents = test_float \
    .filter(col("rating") >= 4.0) \
    .filter(col("user_idx").isin(test_user_idxs_final)) \
    .select("item_idx") \
    .distinct() \
    .count()

print(f"Items pertinents uniques : {items_pertinents:,}")

Users sélectionnés : 100
Users avec produits aimés : 100
Items pertinents uniques : 223


Générer les recommandations pour ces users

In [0]:
# Facteurs pour ces users spécifiques
renamed_users_final = user_factors_v2 \
    .filter(col("id").isin(test_user_idxs_final)) \
    .withColumnRenamed("features", "user_features") \
    .withColumnRenamed("id", "user_idx")

renamed_items_final = item_factors_v2 \
    .withColumnRenamed("features", "item_features") \
    .withColumnRenamed("id", "item_idx")

print(f"Users  : {renamed_users_final.count():,}")
print(f"Items  : {renamed_items_final.count():,}")

# CrossJoin
cross_final = renamed_users_final.crossJoin(renamed_items_final)

scores_final = cross_final.withColumn(
    "score",
    dot_product(F.col("user_features"), F.col("item_features"))
).select("user_idx", "item_idx", "score")

# Top-10 par user
window_spec = Window \
    .partitionBy("user_idx") \
    .orderBy(col("score").desc())

top10_final = scores_final \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .groupBy("user_idx") \
    .agg(collect_list("item_idx").alias("recommended_items"))

print(f" Recs pour {top10_final.count():,} users")

# Vérifier overlap
example = top10_final.join(relevant_final, "user_idx").limit(1).collect()
if example:
    overlap = set(example[0]['recommended_items']) & \
              set(example[0]['relevant_items'])
    print(f"\nRecommended : {example[0]['recommended_items']}")
    print(f"Relevant    : {example[0]['relevant_items'][:10]}")
    print(f"Overlap     : {overlap}")

Users  : 25
Items  : 80,187
 Recs pour 25 users

Recommended : [8969, 5078, 1006, 11639, 22333, 8887, 761, 25785, 54258, 16349]
Relevant    : [26629, 19310, 1561, 281, 19284, 32664, 4870, 13025, 6325, 26834]
Overlap     : set()


Precision@10

In [0]:
precision_final = top10_final.join(relevant_final, "user_idx") \
    .withColumn(
        "precision_at_10",
        size(array_intersect(
            col("recommended_items"),
            col("relevant_items")
        )) / 10
    )

avg_p10_final = precision_final \
    .agg(avg("precision_at_10")) \
    .collect()[0][0]

print("\n" + "=" * 45)
print(" RÉSULTATS FINAUX ALS")
print("=" * 45)
print(f"RMSE         = {rmse_v2:.4f}")
print(f"Precision@10 = {avg_p10_final:.4f}")
print(f"             = {avg_p10_final*100:.1f}%")
print(f"\nObjectif projet : > 15%")


if avg_p10_final > 0.15:
    print(" Objectif atteint !")
elif avg_p10_final > 0.0:
    print(" Résultat positif, sous l'objectif")
    print("→ Normal pour ce dataset très biaisé vers 5 étoiles")
else:
    print(" Encore 0 → problème de mapping")


 RÉSULTATS FINAUX ALS
RMSE         = 2.5304
Precision@10 = 0.0040
             = 0.4%

Objectif projet : > 15%
 Résultat positif, sous l'objectif
→ Normal pour ce dataset très biaisé vers 5 étoiles
